In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(1, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottom = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        b = self.bottom(p4)

        u4 = self.up4(b)
        x4 = self.conv4(torch.cat([u4, d4], dim=1))

        u3 = self.up3(x4)
        x3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(x3)
        x2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(x2)
        x1 = self.conv1(torch.cat([u1, d1], dim=1))

        return torch.sigmoid(self.final(x1))


In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(1, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottom = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        b = self.bottom(p4)

        u4 = self.up4(b)
        x4 = self.conv4(torch.cat([u4, d4], dim=1))

        u3 = self.up3(x4)
        x3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(x3)
        x2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(x2)
        x1 = self.conv1(torch.cat([u1, d1], dim=1))

        return torch.sigmoid(self.final(x1))


In [4]:
import os

images_dir = "../data/train/train_images"
masks_dir = "../data/train/train_masks"

# List files
image_files = sorted(os.listdir(images_dir))
mask_files = sorted(os.listdir(masks_dir))

# Keep only files that exist in BOTH folders
valid_files = [f for f in image_files if f in mask_files]

print("Total matching image-mask pairs:", len(valid_files))
print("Example:", valid_files[:10])


Total matching image-mask pairs: 493
Example: ['101.png', '102.png', '103.png', '104.png', '105.png', '106.png', '107.png', '108.png', '109.png', '110.png']


In [5]:
from PIL import Image
import torch
from torch.utils.data import Dataset
import os
import torchvision.transforms as T

class CariesDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir

        # Default transform if none provided
        self.transform = transform if transform is not None else T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
        ])

        all_images = sorted(os.listdir(images_dir))
        all_masks = sorted(os.listdir(masks_dir))

        # Only matching files
        self.files = [f for f in all_images if f in all_masks]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]

        img_path = os.path.join(self.images_dir, filename)
        mask_path = os.path.join(self.masks_dir, filename)

        img = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        img = self.transform(img)
        mask = self.transform(mask)

        return img, mask


In [6]:
train_images = "../data/train/train_images/"
train_masks  = "../data/train/train_masks/"

dataset = CariesDataset(train_images, train_masks)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print("Training samples:", len(dataset))


Training samples: 493


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet().to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20
best_loss = float("inf")


Device: cuda


In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: True
GPU name: NVIDIA GeForce GTX 1650


In [9]:
save_path = "../models/best_model.pth"

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}")

    # Save BEST model
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), save_path)
        print("💾 Saved new best model!")


Epoch [1/20] Loss: 0.1982
💾 Saved new best model!
Epoch [2/20] Loss: 0.1143
💾 Saved new best model!
Epoch [3/20] Loss: 0.1151
Epoch [4/20] Loss: 0.1144
Epoch [5/20] Loss: 0.1167
Epoch [6/20] Loss: 0.1145
Epoch [7/20] Loss: 0.1149
Epoch [8/20] Loss: 0.1143
💾 Saved new best model!
Epoch [9/20] Loss: 0.1149
Epoch [10/20] Loss: 0.1143
Epoch [11/20] Loss: 0.1151
Epoch [12/20] Loss: 0.1151
Epoch [13/20] Loss: 0.1145
Epoch [14/20] Loss: 0.1141
💾 Saved new best model!
Epoch [15/20] Loss: 0.1153
Epoch [16/20] Loss: 0.1150
Epoch [17/20] Loss: 0.1145
Epoch [18/20] Loss: 0.1151
Epoch [19/20] Loss: 0.1153
Epoch [20/20] Loss: 0.1143


In [10]:
import os

print("Model saved:", os.path.exists(save_path))
print("File size:", os.path.getsize(save_path), "bytes")


Model saved: True
File size: 124139170 bytes
